# 22 · Stacking LightGBM, XGBoost and CatBoost for the average-player model

The average-player model is LightGBM. Would a stack of the strong boosted models do better?

1. **Base models**, each trained on 4 of the 5 match groups and predicting the hidden fifth: LightGBM, XGBoost, CatBoost,
   ExtraTrees and logistic regression with curves (same 41 inputs as the main pipeline).
2. **Six ensembles**: the three boosters with a logistic meta-model; all five with a logistic meta-model; all five plus the
   match situation with a LightGBM meta-model; a plain average of the three boosters; and the three boosters (or two) with an
   **ExtraTrees** meta-model. Judged on unseen-match error, with a 1,000-match bootstrap.
3. **Does it change the batting list?** The best ensemble and CatBoost rebuild the batter skill score and are judged on the
   same 52 internal checks as notebook 11.

**Result.** The best ensemble (average of the three boosters) has 0.08% less error than LightGBM alone: real, but tiny. It picks
the **same 10 batters** and does not predict hidden matches better (0.142 against 0.143), so the simpler single LightGBM stays.
ExtraTrees as the meta-model was worse than LightGBM alone.

**Running it.** Training the five base models takes about an hour on a laptop CPU, so use few threads or a bigger machine.
Their predictions are saved to `outputs/data/research_stacking_oof.npz`, and part 1 reloads them if present, so parts 2 and 3
can be re-run quickly. Parts 2 and 3 use 2 threads and no parallel processes.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

## 1 · Base models (out-of-fold predictions)

<details><summary>Technical notes</summary>

P25 - Full model stacking for the average-player model, with CatBoost (research; nothing existing is changed).  
  
Question (user, 2026-11-15): would stacking the strong boosted models (LightGBM + XGBoost + CatBoost) give a better average  
player than LightGBM alone? Notebook 17 stacked LightGBM, XGBoost, ExtraTrees and logistic but left out CatBoost.  
  
Rule fixed BEFORE running:  
  Base models (same 41 inputs and same 5 match-level folds as the main pipeline; early stopping on a match-level inner  
  split): LightGBM, XGBoost, CatBoost, ExtraTrees, logistic with curves.  
  Stacks, all cross-fitted over the same folds (meta model fitted on 4 folds' out-of-fold predictions, applied to the 5th):  
    A  LightGBM + XGBoost + CatBoost, logistic meta;   B  all five, logistic meta;  
    C  all five + match situation, LightGBM meta;     D  plain average of the three boosters.  
  (Standard stacking: base predictions for the meta training rows come from models that saw the other folds. If anything  
  this flatters the stacks, so a 'no real gain' result is safe.)  
  Judged on unseen-match log-loss; the gain over LightGBM gets a 90% interval from 1,000 match-level bootstraps.  
  The best stack is then pushed through the batter rating (skill score rebuilt from its probabilities) on the same 52  
  internal checks as P9. It would replace LightGBM only if its interval excludes zero AND it wins on the 52 checks.  
  Bowler ratings use the runs value, which needs per-outcome future-run values not stored, so bowlers are not re-run.
</details>

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss
from joblib import Parallel, delayed
from rrlib.common import (CFG, SEED, DATA, load, load_json, load_table, save_table, save_json, get_logger, Timer, weighted_spearman,
                    split_half_reliability, excluded_reasons)
from rrlib import ratings as RL

log = get_logger("p25_stacking_full")
NJ = CFG["n_jobs"]
FAM = load_json("feature_families")
EXO_B = ["wkts_last6", "wkts_last12", "wkts_last24", "balls_since_wkt", "pship_balls", "str_balls", "ns_balls", "free_hit", "rebowl", "bwl_balls"]
FEATS = FAM["A_core"] + FAM["A_derived"] + EXO_B + FAM["C_context"] + ["is_pace"]
LGBP = dict(CFG["models"]["lgb_common"], n_jobs=NJ, random_state=SEED)
NC = 8
CLS = ["dot", "1", "2", "3", "4", "6", "wicket", "wide/nb"]
S = load("features")
S["fold"] = S.match_id.map(load("folds").set_index("match_id").fold).values
y = S.y_class.values
mid = S.match_id.values
CORE_NUM = ["innings", "legal_before", "balls_left", "wkts_before", "score_before", "lvl_code", "year", "venue_par"]
Xl = S[CORE_NUM].copy(); Xl["need"] = S.need.fillna(0); Xl["rrr"] = S.rrr.fillna(0).clip(0, 36)
log.info(f"{len(S)} balls, {S.match_id.nunique()} matches, {len(FEATS)} inputs")


def inner_split(mids_tr, frac=0.15, seed=SEED):
    u = np.unique(mids_tr); r = np.random.RandomState(seed).permutation(u)
    val = set(r[: int(len(u) * frac)])
    return np.array([m in val for m in mids_tr])


BASE = ["LightGBM", "XGBoost", "CatBoost", "ExtraTrees", "Logistic (curves)"]
cache = os.path.join(DATA, "research_stacking_oof.npz")
if os.path.exists(cache):
    z = np.load(cache)
    OOF = {k: z[k.replace(" ", "_").replace("(", "").replace(")", "")] for k in BASE}
    TRAIN_LL = load_json("research_stacking_full_train_ll")
    log.info("loaded cached out-of-fold predictions")
else:
    OOF = {k: np.zeros((len(S), NC)) for k in BASE}
    TRAIN_LL = {k: [] for k in BASE}
    for k in range(5):
        tr, te = (S.fold != k).values, (S.fold == k).values
        Xtr, Xte, ytr, mtr = S.loc[tr, FEATS], S.loc[te, FEATS], y[tr], mid[tr]
        m = inner_split(mtr)
        with Timer(log, f"fold {k}: LightGBM"):
            d1 = lgb.Dataset(Xtr[~m], ytr[~m]); d2 = lgb.Dataset(Xtr[m], ytr[m])
            b = lgb.train(dict(LGBP, objective="multiclass", num_class=NC, learning_rate=0.06), d1, 3000, valid_sets=[d2],
                          callbacks=[lgb.early_stopping(50, verbose=False)])
            OOF["LightGBM"][te] = b.predict(Xte, num_iteration=b.best_iteration)
            TRAIN_LL["LightGBM"].append(log_loss(ytr, b.predict(Xtr, num_iteration=b.best_iteration), labels=range(NC)))
        with Timer(log, f"fold {k}: XGBoost"):
            xm = xgb.XGBClassifier(objective="multi:softprob", num_class=NC, n_estimators=3000, learning_rate=0.06, max_depth=6,
                                   min_child_weight=50, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, tree_method="hist",
                                   n_jobs=NJ, random_state=SEED, early_stopping_rounds=50, eval_metric="mlogloss")
            xm.fit(Xtr[~m], ytr[~m], eval_set=[(Xtr[m], ytr[m])], verbose=False)
            OOF["XGBoost"][te] = xm.predict_proba(Xte)
            TRAIN_LL["XGBoost"].append(log_loss(ytr, xm.predict_proba(Xtr), labels=range(NC)))
        with Timer(log, f"fold {k}: CatBoost"):
            cb = CatBoostClassifier(loss_function="MultiClass", iterations=3000, learning_rate=0.08, depth=6, l2_leaf_reg=5,
                                    random_seed=SEED, thread_count=NJ, od_type="Iter", od_wait=50, verbose=False)
            cb.fit(Xtr[~m], ytr[~m], eval_set=(Xtr[m], ytr[m]), use_best_model=True)
            OOF["CatBoost"][te] = cb.predict_proba(Xte)
            TRAIN_LL["CatBoost"].append(log_loss(ytr, cb.predict_proba(Xtr), labels=range(NC)))
        with Timer(log, f"fold {k}: ExtraTrees + logistic"):
            et = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=100, max_features=0.5, n_jobs=NJ, random_state=SEED)
            et.fit(Xtr.fillna(-1), ytr)
            OOF["ExtraTrees"][te] = np.clip(et.predict_proba(Xte.fillna(-1)), 1e-6, 1)
            TRAIN_LL["ExtraTrees"].append(log_loss(ytr, np.clip(et.predict_proba(Xtr.fillna(-1)), 1e-6, 1), labels=range(NC)))
            sub = np.random.RandomState(SEED).choice(np.where(tr)[0], min(60000, tr.sum()), replace=False)
            lr = make_pipeline(StandardScaler(), SplineTransformer(n_knots=5, degree=3), LogisticRegression(max_iter=400, C=1.0))
            lr.fit(Xl.iloc[sub], y[sub])
            pr = np.zeros((te.sum(), NC)); pr[:, lr[-1].classes_] = lr.predict_proba(Xl[te]); OOF["Logistic (curves)"][te] = np.clip(pr, 1e-6, 1)
            TRAIN_LL["Logistic (curves)"].append(log_loss(y[sub], np.clip(lr.predict_proba(Xl.iloc[sub]), 1e-6, 1), labels=range(NC)))
    for k_ in OOF:
        OOF[k_] = OOF[k_] / OOF[k_].sum(1, keepdims=True)
    np.savez_compressed(cache, **{k.replace(" ", "_").replace("(", "").replace(")", ""): OOF[k] for k in BASE})
    save_json({k: float(np.mean(v)) for k, v in TRAIN_LL.items()}, "research_stacking_full_train_ll")
    TRAIN_LL = {k: float(np.mean(v)) for k, v in TRAIN_LL.items()}


01:30:46 200961 balls, 867 matches, 41 inputs


01:30:46 loaded cached out-of-fold predictions


## 2 · Six ensembles, judged on unseen matches

<details><summary>Technical notes</summary>

P25b - Meta models on the saved P25 base predictions (light: no base model is retrained). Research only.  
  
P25 trained LightGBM, XGBoost, CatBoost, ExtraTrees and logistic on 5 match-level folds and saved their out-of-fold  
predictions (outputs/data/research_stacking_oof.npz) before the laptop crashed in a later, parallel step.  
This script only fits the meta models, with 2 threads and no parallel processes.  
Stacks (cross-fitted over the same 5 folds):  
  A  LightGBM + XGBoost + CatBoost, logistic meta        B  all five, logistic meta  
  C  all five + match situation, LightGBM meta          D  plain average of the three boosters  
  E  LightGBM + XGBoost + CatBoost, ExtraTrees meta     F  LightGBM + XGBoost, ExtraTrees meta   (user's request)  
Rule fixed before running: a stack is taken further only if its gain over LightGBM has a 90% match-level bootstrap  
interval that excludes zero; only then is it pushed through the batting rating on the 52 internal checks (sequentially).
</details>

In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from rrlib.common import CFG, SEED, DATA, load, load_json, load_table, save_table, save_json, get_logger, Timer
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

log = get_logger("p25b_stacking_meta")
NC, NJ = 8, 2
S = load("features", columns=["ball_uid", "match_id", "y_class", "innings", "legal_before", "balls_left", "wkts_before", "score_before",
                              "lvl_code", "year", "venue_par", "need", "rrr"])
S["fold"] = S.match_id.map(load("folds").set_index("match_id").fold).values
y, mid = S.y_class.values, S.match_id.values
Xl = S[["innings", "legal_before", "balls_left", "wkts_before", "score_before", "lvl_code", "year", "venue_par"]].copy()
Xl["need"] = S.need.fillna(0); Xl["rrr"] = S.rrr.fillna(0).clip(0, 36)
BASE = ["LightGBM", "XGBoost", "CatBoost", "ExtraTrees", "Logistic (curves)"]
z = np.load(os.path.join(DATA, "research_stacking_oof.npz"))
OOF = {k: z[k.replace(" ", "_").replace("(", "").replace(")", "")] for k in BASE}
TRAIN_LL = load_json("research_stacking_full_train_ll")
assert all(v_.shape == (len(S), NC) for v_ in OOF.values())


def inner_split(mids_tr, frac=0.15, seed=SEED):
    u = np.unique(mids_tr); r = np.random.RandomState(seed).permutation(u)
    val = set(r[: int(len(u) * frac)])
    return np.array([m in val for m in mids_tr])


LP = lambda keys: np.hstack([np.log(np.clip(OOF[k], 1e-6, 1)) for k in keys])
PR = lambda keys: np.hstack([OOF[k] for k in keys])


def cross_fit(make, Z):
    st = np.zeros((len(S), NC))
    for k in range(5):
        tr, te = (S.fold != k).values, (S.fold == k).values
        st[te] = make(Z[tr], y[tr], mid[tr]).predict_proba(Z[te])
    return np.clip(st, 1e-6, 1) / np.clip(st, 1e-6, 1).sum(1, keepdims=True)


class LGBMeta:
    def __init__(self, Z, yy, mm):
        m = inner_split(mm)
        p = dict(CFG["models"]["lgb_common"], n_jobs=NJ, random_state=SEED, objective="multiclass", num_class=NC, learning_rate=0.05, num_leaves=15)
        self.b = lgb.train(p, lgb.Dataset(Z[~m], yy[~m]), 2000, valid_sets=[lgb.Dataset(Z[m], yy[m])], callbacks=[lgb.early_stopping(50, verbose=False)])

    def predict_proba(self, Z):
        return self.b.predict(Z, num_iteration=self.b.best_iteration)


logit = lambda Z, yy, mm: LogisticRegression(max_iter=600, C=1.0).fit(Z, yy)
etree = lambda Z, yy, mm: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=200, max_features="sqrt", n_jobs=NJ, random_state=SEED).fit(Z, yy)
P = dict(OOF)
with Timer(log, "meta models"):
    P["Stack A: LightGBM + XGBoost + CatBoost (logistic meta)"] = cross_fit(logit, LP(["LightGBM", "XGBoost", "CatBoost"]))
    P["Stack B: all five (logistic meta)"] = cross_fit(logit, LP(BASE))
    P["Stack C: all five + situation (LightGBM meta)"] = cross_fit(LGBMeta, np.hstack([LP(BASE), Xl.values]))
    P["Average of LightGBM + XGBoost + CatBoost"] = (OOF["LightGBM"] + OOF["XGBoost"] + OOF["CatBoost"]) / 3
    P["Stack E: LightGBM + XGBoost + CatBoost (ExtraTrees meta)"] = cross_fit(etree, PR(["LightGBM", "XGBoost", "CatBoost"]))
    P["Stack F: LightGBM + XGBoost (ExtraTrees meta)"] = cross_fit(etree, PR(["LightGBM", "XGBoost"]))

lookup_ll = float(load_table("m1_model_comparison").set_index("model").multi_logloss["Lookup table (innings x phase x wickets x level)"])
ball_ll = {k: -np.log(np.clip(Pk[np.arange(len(y)), y], 1e-12, 1)) for k, Pk in P.items()}
codes, uniq = pd.factorize(mid)
n_m = np.bincount(codes)
rng = np.random.RandomState(SEED)
boots = [np.bincount(rng.randint(0, len(uniq), len(uniq)), minlength=len(uniq)) for _ in range(1000)]
base_sum = np.bincount(codes, weights=ball_ll["LightGBM"])
L0 = float(ball_ll["LightGBM"].mean())
rows = []
for k_, Pk in P.items():
    ll = float(ball_ll[k_].mean())
    d_sum = np.bincount(codes, weights=ball_ll[k_]) - base_sum
    bs = np.array([(w * d_sum).sum() / (w * n_m).sum() for w in boots])
    cal = [abs((y == i).mean() - Pk[:, i].mean()) for i in range(NC)]
    rows.append({"model": k_, "unseen_logloss": ll, "train_logloss": TRAIN_LL.get(k_, np.nan),
                 "pct_less_error_than_lookup": 100 * (1 - ll / lookup_ll), "pct_less_error_than_lightgbm": 100 * (1 - ll / L0),
                 "diff_lo90": float(np.quantile(bs, 0.05)), "diff_hi90": float(np.quantile(bs, 0.95)),
                 "worst_class_rate_error_pct": 100 * max(cal), "top1_hit": float((Pk.argmax(1) == y).mean())})
R = pd.DataFrame(rows).sort_values("unseen_logloss").reset_index(drop=True)
R["gap"] = R.unseen_logloss - R.train_logloss
R["gain_is_real"] = R.diff_hi90 < 0
save_table(R, "research_stacking_full")
pd.set_option("display.width", 250)
log.info("\n" + R.drop(columns=["top1_hit"]).round(5).to_string())
stk = R[R.model.str.startswith(("Stack", "Average"))]
summ = {"best_stack": stk.iloc[0].model, "best_stack_pct_less_error_than_lightgbm": float(stk.iloc[0].pct_less_error_than_lightgbm),
        "any_stack_gain_is_real": bool(stk.gain_is_real.any()), "lookup_logloss": lookup_ll}
summ["decision"] = ("take further: " + ", ".join(stk[stk.gain_is_real].model)) if summ["any_stack_gain_is_real"] else "keep LightGBM (no stack's gain is clear of noise)"
save_json(summ, "research_stacking_full_summary")
log.info(summ)


01:41:18 [meta models] 631.3s


01:41:19 
                                                       model  unseen_logloss  train_logloss  pct_less_error_than_lookup  pct_less_error_than_lightgbm  diff_lo90  diff_hi90  worst_class_rate_error_pct      gap  gain_is_real
0                   Average of LightGBM + XGBoost + CatBoost         1.57677            NaN                     1.91442                       0.08406   -0.00150   -0.00116                     0.03820      NaN          True
1     Stack A: LightGBM + XGBoost + CatBoost (logistic meta)         1.57687            NaN                     1.90860                       0.07813   -0.00153   -0.00094                     0.02110      NaN          True
2                          Stack B: all five (logistic meta)         1.57716            NaN                     1.89036                       0.05955   -0.00127   -0.00061                     0.04570      NaN          True
3                                                   CatBoost         1.57721        1.55632       

01:41:19 {'best_stack': 'Average of LightGBM + XGBoost + CatBoost', 'best_stack_pct_less_error_than_lightgbm': 0.08406093419717209, 'any_stack_gain_is_real': True, 'lookup_logloss': 1.6075481421803306, 'decision': 'take further: Average of LightGBM + XGBoost + CatBoost, Stack A: LightGBM + XGBoost + CatBoost (logistic meta), Stack B: all five (logistic meta)'}


## 3 · Does it change or improve the batting list?

<details><summary>Technical notes</summary>

P25c - Does the small, real gain from stacking change or improve the batting list? (research, light, sequential)  
  
P25b found that the plain average of LightGBM + XGBoost + CatBoost has 0.08% less unseen error than LightGBM, and CatBoost  
alone 0.06% (both clear of noise). By the rule fixed in P25b, the best stack is pushed through the batter rating.  
Each model's out-of-fold probabilities rebuild the batter skill score exactly as P5 does (residuals for boundary, dot and  
dismissal; run values by regression on V-1; weights by split-half repeatability). All are re-run here the same way,  
including LightGBM, so the comparison is like for like. Judged on the same 52 internal checks as P9 (and 2026 for  
information). The average-player model would change only if a version beats LightGBM on the 52 checks.  
Runs sequentially on 2 threads (the laptop crashed when the parallel version ran).
</details>

In [4]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from rrlib.common import CFG, SEED, DATA, load, load_json, load_table, save_table, save_json, get_logger, Timer, weighted_spearman, \
    split_half_reliability, excluded_reasons
from rrlib import ratings as RL
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

log = get_logger("p25c_stacking_downstream")
z = np.load(os.path.join(DATA, "research_stacking_oof.npz"))
MODELS = {"LightGBM": z["LightGBM"], "CatBoost": z["CatBoost"],
          "Average of LightGBM + XGBoost + CatBoost": (z["LightGBM"] + z["XGBoost"] + z["CatBoost"]) / 3}
v = load("values")
yc = load("features", columns=["ball_uid", "y_class"])
assert (v.ball_uid.values == yc.ball_uid.values).all()
y = yc.y_class.values
faced = v.faced.values.astype(bool)
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]["bat"]
cols = ["r_bdry_bat", "r_dot_bat", "r_wkt_bat"]
V2 = v.copy()
tags = {}
for i, (name, Pm) in enumerate(MODELS.items()):
    pb = Pm[:, :7] / Pm[:, :7].sum(1, keepdims=True)
    d = pd.DataFrame({"r_bdry_bat": np.isin(y, [4, 5]) - (pb[:, 4] + pb[:, 5]), "r_dot_bat": (y == 0) - pb[:, 0],
                      "r_wkt_bat": v.striker_out.values.astype(float) - pb[:, 6]})
    d.loc[~faced] = np.nan
    w = d.assign(v1_bat=v.v1_bat.values, striker_id=v.striker_id.values, match_id=v.match_id.values).dropna()
    beta = dict(zip(cols, LinearRegression(fit_intercept=False).fit(w[cols], w["v1_bat"]).coef_))
    rel = {c: max(0.0, split_half_reliability(w, "striker_id", "match_id", c, min_n=60)[0]) for c in cols}
    tag = f"v{7 + i}"
    V2[f"{tag}_bat"] = np.where(faced, sum(rel[c] * beta[c] * d[c].values for c in cols), np.nan)
    tags[name] = tag
    log.info(f"{name}: weights {({c: round(rel[c] * beta[c], 3) for c in cols})}")

HOLD = CFG["cv"]["holdout_season"]
VV = V2[V2.year < HOLD]
splits = [("rolling", f"{a[0]}-{a[-1]}->{b_}", VV.year.isin(a).values, (VV.year == b_).values) for a, b_ in CFG["cv"]["rolling"]]
mids_ = np.sort(VV.match_id.unique()); rg = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rg.choice(mids_, len(mids_) // 2, replace=False)); mm = VV.match_id.isin(a).values
    splits.append(("random", f"r{i}", mm, ~mm))
splits.append(("holdout", "2022-25->2026", (V2.year < HOLD).values, (V2.year == HOLD).values))
rows = []
with Timer(log, "52 checks + 2026, sequential"):
    for kind, sname, tr_m, te_m in splits:
        base = V2 if kind == "holdout" else VV
        tr, te = base[tr_m], base[te_m]
        t = te[te.is_pressure & te.v2_bat.notna()]
        yy = t.groupby("striker_id").agg(y2=("v2_bat", "mean"), y1=("v1_bat", "mean"), n=("v2_bat", "size")); yy = yy[yy.n >= 20]
        pool = yy.index[yy.index.isin(set(tr.loc[tr.is_pressure & tr.v1_bat.notna(), "striker_id"]))]
        if len(pool) < 15:
            continue
        for name, tag in tags.items():
            sc = RL.build_scores(tr, "bat", dict(chosen, value=tag), comp).score.reindex(pool).fillna(-1e9)
            top = sc.sort_values(ascending=False).index[:10]
            wm = lambda col, idx: np.average(yy.loc[idx, col], weights=yy.loc[idx, "n"])
            rows.append({"kind": kind, "split": sname, "model": name,
                         "rho": weighted_spearman(sc.values, yy.loc[pool].y2.values, yy.loc[pool].n.values),
                         "top10_lift_runs": 100 * (wm("y1", top) - wm("y1", pool))})
CV = pd.DataFrame(rows)
save_table(CV, "research_stacking_downstream_cv")
summ = {}
for name in tags:
    r = CV[(CV.model == name) & (CV.kind != "holdout")]
    summ[name] = {"objective": float(0.5 * r[r.kind == "random"].rho.mean() + 0.5 * r[r.kind == "rolling"].rho.mean()),
                  "rolling": float(r[r.kind == "rolling"].rho.mean()), "top10_lift_runs": float(r.top10_lift_runs.mean()),
                  "holdout_2026": float(CV[(CV.model == name) & (CV.kind == "holdout")].rho.iloc[0])}
piv = CV[CV.kind == "random"].pivot(index="split", columns="model", values="rho")
for name in tags:
    if name == "LightGBM":
        continue
    dd = piv[name] - piv["LightGBM"]
    summ[name].update({"vs_lightgbm_diff": float(dd.mean()), "vs_lightgbm_se": float(dd.std(ddof=1) / np.sqrt(len(dd))),
                       "better_splits": int((dd > 0).sum())})
EX = set(excluded_reasons()); names = load("players").set_index("player_id").name
submitted = load_table("final_top30_bat").head(10).name.tolist()
for name, tag in tags.items():
    sc = RL.build_scores(V2, "bat", dict(chosen, value=tag), comp)
    sc = sc[~sc.index.isin(EX)].sort_values("score", ascending=False)
    top = [names.get(p) for p in sc.index[:10]]
    summ[name]["top10"] = top
    summ[name]["same_names_as_submitted"] = len(set(top) & set(submitted))
best = max((n for n in tags if n != "LightGBM"), key=lambda n: summ[n]["objective"])
gain = summ[best]["objective"] - summ["LightGBM"]["objective"]
summ["decision"] = (f"switch to {best}" if gain > summ[best]["vs_lightgbm_se"] and summ[best]["rolling"] >= summ["LightGBM"]["rolling"]
                    else "keep LightGBM")
save_json(summ, "research_stacking_downstream_summary")
log.info("\n" + pd.DataFrame({k: {kk: vv for kk, vv in v_.items() if kk != "top10"} for k, v_ in summ.items() if isinstance(v_, dict)}).round(4).to_string())
for name in tags:
    log.info(f"{name} top 10: {summ[name]['top10']}")
log.info(f"decision by the pre-registered rule: {summ['decision']}")


01:41:19 LightGBM: weights {'r_bdry_bat': np.float64(1.621), 'r_dot_bat': np.float64(-0.508), 'r_wkt_bat': np.float64(-1.404)}


01:41:20 CatBoost: weights {'r_bdry_bat': np.float64(1.616), 'r_dot_bat': np.float64(-0.503), 'r_wkt_bat': np.float64(-1.442)}


01:41:20 Average of LightGBM + XGBoost + CatBoost: weights {'r_bdry_bat': np.float64(1.609), 'r_dot_bat': np.float64(-0.503), 'r_wkt_bat': np.float64(-1.387)}


01:41:44 [52 checks + 2026, sequential] 24.2s


01:41:45 
                         LightGBM  CatBoost  Average of LightGBM + XGBoost + CatBoost
objective                  0.1427    0.1386                                    0.1421
rolling                    0.1500    0.1436                                    0.1490
top10_lift_runs           11.6837   11.5516                                   11.6613
holdout_2026              -0.0861   -0.0863                                   -0.0869
same_names_as_submitted   10.0000   10.0000                                   10.0000
vs_lightgbm_diff              NaN   -0.0018                                   -0.0003
vs_lightgbm_se                NaN    0.0003                                    0.0002
better_splits                 NaN   12.0000                                   23.0000


01:41:45 LightGBM top 10: ['Abhishek Sharma', 'Vaibhav Sooryavanshi', 'Suryakumar Yadav', 'Phil Salt', 'Tim David', 'Yashasvi Jaiswal', 'Heinrich Klaasen', 'Rovman Powell', 'Josh Inglis', 'Priyansh Arya']


01:41:45 CatBoost top 10: ['Abhishek Sharma', 'Vaibhav Sooryavanshi', 'Suryakumar Yadav', 'Phil Salt', 'Heinrich Klaasen', 'Tim David', 'Yashasvi Jaiswal', 'Rovman Powell', 'Josh Inglis', 'Priyansh Arya']


01:41:45 Average of LightGBM + XGBoost + CatBoost top 10: ['Abhishek Sharma', 'Vaibhav Sooryavanshi', 'Suryakumar Yadav', 'Phil Salt', 'Tim David', 'Heinrich Klaasen', 'Yashasvi Jaiswal', 'Rovman Powell', 'Josh Inglis', 'Priyansh Arya']


01:41:45 decision by the pre-registered rule: keep LightGBM
